[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part3_agents/ch08_agent_basics.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 8장 AI 에이전트의 원리
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제3부 에이전트 — 스스로 일하는 LLM
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
# 실습 라이브러리 설치 (검증 시점 2026-06, 하한 고정)
!pip install -q "google-genai>=1.0"

In [ ]:
# 설치된 핵심 라이브러리 버전 기록 (재현용) — 이 출력의 버전을 그대로 고정하면 가장 안정적이다
import importlib.metadata as _m
for _p in ["google-genai","langchain","langchain-google-genai","langchain-community",
           "langchain-text-splitters","langgraph","faiss-cpu","tiktoken","rank-bm25",
           "networkx","tavily-python"]:
    try: print(f"{_p}=={_m.version(_p)}")
    except Exception: pass

### API 키와 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.5-flash"   # 모델 갱신 시 이 한 줄만 변경

### 이전 장에서 이어지는 준비

In [ ]:
# 이전 장에서 이어지는 준비 (4·5장: 임베딩과 검색)
import numpy as np
EMBED_MODEL = "text-embedding-004"
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
def embed(text):
    return client.models.embed_content(model=EMBED_MODEL, contents=text).embeddings[0].values
docs = ["환불은 구매 후 7일 이내에 가능하다",
        "연차는 입사 1년 후 15일이 주어진다",
        "재택근무는 주 2회까지 가능하다"]
doc_vecs = [embed(d) for d in docs]
def retrieve(query, k=2):
    q = embed(query)
    return [d for d, _ in sorted(zip(docs, doc_vecs), key=lambda x: cosine(q, x[1]), reverse=True)[:k]]

## 예제 8-1. ReAct 에이전트 루프

In [ ]:
import json

def calculator(expression):
    return eval(expression)   # 실습용. 실제로는 안전한 수식 해석기를 쓴다

schema = {
    "type": "object",
    "properties": {
        "생각": {"type": "string"},
        "행동": {"type": "string"},   # calculate 또는 answer
        "입력": {"type": "string"},
    },
    "required": ["생각", "행동", "입력"],
}

def react_agent(question, max_steps=5):
    scratch = ""
    for step in range(max_steps):          # 최대 단계 = 무한 루프 안전장치
        prompt = (
            f"질문: {question}\n지금까지 계산: {scratch}\n"
            f"계산이 필요하면 행동을 calculate로, 답이 준비되면 answer로 정하라."
        )
        res = client.models.generate_content(
            model=GEMINI_FLASH,
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": schema,
                "temperature": 0,
            },
        )
        decision = json.loads(res.text)
        print(f"[생각] {decision['생각']}")
        if decision["행동"] == "answer":
            return decision["입력"]
        result = calculator(decision["입력"])
        print(f"[행동] calculate({decision['입력']}) = {result}")
        scratch += f" {decision['입력']}={result}"
    return "최대 단계 초과"

print("답:", react_agent("사과 3개에 1200원, 배 2개에 1800원이면 모두 얼마?"))

## 예제 8-2. RAG를 장기 기억 도구로

In [ ]:
def search_memory(query):       # 2부의 RAG가 에이전트의 장기 기억이 된다
    return retrieve(query, k=1)[0]

print(search_memory("환불 며칠 안에 돼?"))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 8장 노트북